In [ ]:
!pip install nvcc4jupyter

In [ ]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp38c5rb_4".


In [ ]:
%%cuda
#include "cuda_runtime.h"
#include "device_launch_parameters.h"

#include <stdio.h>
#include <stdlib.h>
#include <math.h>

// . . .
// defines
#define N 1024
// kernel
__global__ void pi(float* suma)
{
	// Reserva de espacio en la zona de memoria compartida
	__shared__ float temporal[N];
	// Indice local de cada hilo -> kernel con un solo bloque
	int myID = threadIdx.x;
	// Generamos en 'temporal' el valor de la serie
	int D = myID + 1;
	temporal[myID] = 1.0 / (D * D);
	__syncthreads();
	//Reduccion paralela
	int salto = N / 2;
	// Realizamos log2(N) iteraciones
	while (salto)
	{
		// Solo trabajan la mitad de los hilos
		if (myID < salto)
		{
			temporal[myID] = temporal[myID] + temporal[myID + salto];
		}
		__syncthreads();
		salto = salto / 2;
	}
	// El hilo no.'0' escribe el resultado final en la memoria global
	if (myID == 0)
	{
		suma[0] = temporal[myID];
	}
}


int main()
{
	// declaraciones
	float* suma;
	float* dev_suma;
	// reserva en el host
	suma = (float*)malloc(sizeof(float));
	// reserva en el device
	cudaMalloc((void**)&dev_suma, sizeof(float));

	// lanzamiento del kernel
	printf("Calcular el valor de Pi en base %d términos\n", N);
	pi << < 1, N >> > (dev_suma);

	// recogida de datos desde el device
	cudaMemcpy(suma, dev_suma, sizeof(float), cudaMemcpyDeviceToHost);

	// calcular el  valor de pi
	float ValorPi = sqrt(suma[0] * 6);
	// impresion de resultados
	printf("Resultado: %.20f\n", ValorPi);

	// liberamos memoria en el device
	cudaFree(suma);
	// salida
	printf("\npulsa INTRO para finalizar...");
	fflush(stdin);
	char tecla = getchar();
	return 0;
}


Calcular el valor de Pi en base 1024 términos
Resultado: 3.14066076278686523438

pulsa INTRO para finalizar...
